# AIME Trace Generation on H100

Generate TIR traces with gpt-oss-120b for offline analysis.
- 8 samples per problem with logprobs
- Saves traces as JSON for download
- Use remaining H100 quota (30 hours/week)

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess, json, re, math, time, queue, threading, contextlib
from pathlib import Path

In [ ]:
def set_env(archive, tmp):
    if not os.path.exists(tmp):
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', tmp], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', f'{tmp}/wheels',
                    'unsloth', 'trl', 'vllm', 'openai_harmony'], check=True)

set_env('/kaggle/input/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

In [ ]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false'), ('TRITON_PTXAS_PATH', '/usr/local/cuda/bin/ptxas'),
             ('TIKTOKEN_ENCODINGS_BASE', '/kaggle/tmp/setup/tiktoken_encodings')]:
    os.environ[k] = v

In [ ]:
from jupyter_client import KernelManager
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from openai import OpenAI
from openai_harmony import (HarmonyEncodingName, load_harmony_encoding, SystemContent, ReasoningEffort,
                             ToolNamespaceConfig, Author, Message, Role, TextContent, Conversation)
from transformers import set_seed

## Configuration

In [ ]:
class CFG:
    # Model
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    kv_cache_dtype, dtype = 'fp8_e4m3', 'auto'
    
    # Trace generation
    n_samples = 8
    max_turns = 12
    temperature = 0.7
    top_logprobs = 5
    
    # Prompts
    system_prompt = ('You are a world-class International Mathematical Olympiad (IMO) competitor. '
                    'The final answer must be a non-negative integer between 0 and 99999. '
                    'You must place the final integer answer inside \\boxed{}.')
    tool_prompt = ('Use this tool to execute Python code. The environment is a stateful Jupyter notebook. '
                  'You must use print() to output results.')
    preference_prompt = 'You have access to `math`, `numpy` and `sympy` to solve the problem.'
    
    # Timing
    server_timeout = 180
    sample_timeout = 300  # 5 min per sample max
    jupyter_timeout = 6
    
    # vLLM
    context_tokens = 65536
    buffer_tokens = 512
    gpu_memory_utilization = 0.96
    batch_size = 256
    min_p = 0.02
    seed = 42
    
    # Output
    output_dir = '/kaggle/working/traces'

set_seed(CFG.seed)
os.makedirs(CFG.output_dir, exist_ok=True)
print(f"Generating {CFG.n_samples} samples per problem with {CFG.max_turns} max turns")

## Load AIME Problems

In [ ]:
# AIME 2023-2024 Test Set (20 problems for trace generation)
AIME_PROBLEMS = [
    {"id": "2023-I-1", "problem": "Five men and nine women stand equally spaced around a circle in random order. The probability that every man stands diametrically opposite a woman is $\\frac{m}{n},$ where $m$ and $n$ are relatively prime positive integers. Find $m+n.$", "answer": 191},
    {"id": "2023-I-2", "problem": "Positive real numbers $b \\not= 1$ and $n$ satisfy the equations $\\sqrt{\\log_b n} = \\log_b \\sqrt{n}$ and $b \\cdot \\log_b n = \\log_b (bn).$ The value of $n$ is $\\frac{j}{k},$ where $j$ and $k$ are relatively prime positive integers. Find $j+k.$", "answer": 881},
    {"id": "2023-I-3", "problem": "A plane contains $40$ lines, no $2$ of which are parallel. Suppose that there are $3$ points where exactly $3$ lines intersect, $4$ points where exactly $4$ lines intersect, $5$ points where exactly $5$ lines intersect, $6$ points where exactly $6$ lines intersect, and no points where more than $6$ lines intersect. Find the number of points where exactly $2$ lines intersect.", "answer": 607},
    {"id": "2023-I-4", "problem": "The sum of all positive integers $m$ such that $\\frac{13!}{m}$ is a perfect square can be written as $2^a3^b5^c7^d11^e13^f,$ where $a,b,c,d,e,$ and $f$ are positive integers. Find $a+b+c+d+e+f.$", "answer": 12},
    {"id": "2023-I-5", "problem": "Let $P$ be a point on the circle circumscribing square $ABCD$ that satisfies $PA \\cdot PC = 56$ and $PB \\cdot PD = 90.$ Find the area of $ABCD.$", "answer": 106},
    {"id": "2023-I-6", "problem": "Alice knows that $3$ red cards and $3$ black cards will be revealed to her one at a time in random order. Before each card is revealed, Alice must guess its color. If Alice plays optimally, the expected number of cards she will guess correctly is $\\frac{m}{n},$ where $m$ and $n$ are relatively prime positive integers. Find $m+n.$", "answer": 51},
    {"id": "2023-I-7", "problem": "Call a positive integer $n$ extra-distinct if the remainders when $n$ is divided by $2, 3, 4, 5,$ and $6$ are distinct. Find the number of extra-distinct positive integers less than $1000.$", "answer": 49},
    {"id": "2023-I-8", "problem": "Rhombus $ABCD$ has $\\angle BAD < 90^\\circ.$ There is a point $P$ on the incircle of the rhombus such that the distances from $P$ to the lines $DA,AB,$ and $BC$ are $9,5,$ and $16,$ respectively. Find the perimeter of $ABCD.$", "answer": 125},
    {"id": "2023-I-9", "problem": "Find the number of cubic polynomials $p(x) = x^3 + ax^2 + bx + c,$ where $a, b,$ and $c$ are integers in $\\{-20,-19,-18,\\ldots,18,19,20\\},$ such that there is a unique integer $m \\not= 2$ with $p(m) = p(2).$", "answer": 738},
    {"id": "2023-I-10", "problem": "There exists a unique positive integer $a$ for which the sum $U=\\sum_{n=1}^{2023}\\left\\lfloor\\dfrac{n^{2}-na}{5}\\right\\rfloor$ is an integer strictly between $-1000$ and $1000.$ For that unique $a,$ find $a+U.$", "answer": 944},
    {"id": "2023-II-1", "problem": "The numbers of apples growing on each of six apple trees form an arithmetic sequence where the greatest number of apples growing on any of the six trees is double the least number of apples growing on any of the six trees. The total number of apples growing on all six trees is $990.$ Find the greatest number of apples growing on any of the six trees.", "answer": 220},
    {"id": "2023-II-2", "problem": "Recall that a palindrome is a number that reads the same forward and backward. Find the greatest integer less than $1000$ that is a palindrome both when written in base ten and when written in base eight, such as $292 = 444_{\\text{eight}}.$", "answer": 585},
    {"id": "2023-II-3", "problem": "Let $\\triangle ABC$ be an isosceles triangle with $\\angle A = 90^\\circ.$ There exists a point $P$ inside $\\triangle ABC$ such that $\\angle PAB = \\angle PBC = \\angle PCA$ and $AP = 10.$ Find the area of $\\triangle ABC.$", "answer": 250},
    {"id": "2023-II-4", "problem": "Let $x,y,$ and $z$ be real numbers satisfying the system $xy + 4z = 60,$ $yz + 4x = 60,$ $zx + 4y = 60.$ Let $S$ be the set of possible values of $x.$ Find the sum of the squares of the elements of $S.$", "answer": 273},
    {"id": "2023-II-5", "problem": "Let $S$ be the set of all positive rational numbers $r$ such that when the two numbers $r$ and $55r$ are written as fractions in lowest terms, the sum of the numerator and denominator of one fraction is the same as the sum of the numerator and denominator of the other fraction. The sum of all the elements of $S$ can be expressed in the form $\\frac{p}{q},$ where $p$ and $q$ are relatively prime positive integers. Find $p+q.$", "answer": 719},
    {"id": "2024-I-1", "problem": "Every morning Aya goes for a $9$-kilometer-long walk and stops at a coffee shop afterwards. When she walks at a constant speed of $s$ kilometers per hour, the walk takes her 4 hours, including $t$ minutes spent in the coffee shop. When she walks $s+2$ kilometers per hour, the walk takes her 2 hours and 24 minutes, including $t$ minutes spent in the coffee shop. Suppose Aya walks at $s+\\frac{1}{2}$ kilometers per hour. Find the number of minutes the walk takes her, including the $t$ minutes spent in the coffee shop.", "answer": 204},
    {"id": "2024-I-2", "problem": "There exist real numbers $x$ and $y$, both greater than 1, such that $\\log_x\\left(y^x\\right)=\\log_y\\left(x^{4y}\\right)=10$. Find $xy$.", "answer": 25},
    {"id": "2024-I-3", "problem": "Alice and Bob play the following game. A stack of $n$ tokens lies before them. The players take turns with Alice going first. On each turn, the player removes either $1$ token or $4$ tokens from the stack. Whoever removes the last token wins. Find the number of positive integers $n$ less than or equal to $2024$ for which there exists a strategy for Bob that guarantees that Bob will win the game regardless of Alice's play.", "answer": 809},
    {"id": "2024-I-4", "problem": "Jen enters a lottery by picking $4$ distinct numbers from $S=\\{1,2,3,\\cdots,9,10\\}.$ $4$ numbers are randomly chosen from $S.$ She wins a prize if at least two of her numbers were $2$ of the randomly chosen numbers, and wins the grand prize if all four of her numbers were the randomly chosen numbers. The probability that she wins a prize is $\\frac{m}{n}$ where $m$ and $n$ are coprime. Find $m+n$.", "answer": 116},
    {"id": "2024-I-5", "problem": "Rectangles $ABCD$ and $EFGH$ are drawn such that $D,E,C,F$ are collinear. Also, $A,D,H,G$ all lie on a circle. If $BC=16$, $AB=107$, $FG=17$, and $EF=184$, what is the length of $CE$?", "answer": 104},
]

print(f"Loaded {len(AIME_PROBLEMS)} problems (AIME 2023-2024 test set)")

## Sandbox and Tool Classes

In [ ]:
class AIMO3Template:
    def get_system_content(self, prompt, tool_cfg):
        return SystemContent.new().with_model_identity(prompt).with_reasoning_effort(
            reasoning_effort=ReasoningEffort.HIGH).with_tools(tool_cfg)

    def apply_chat_template(self, sys_prompt, usr_prompt, tool_cfg):
        return [Message.from_role_and_content(Role.SYSTEM, self.get_system_content(sys_prompt, tool_cfg)),
                Message.from_role_and_content(Role.USER, usr_prompt)]

In [ ]:
class AIMO3Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout):
        self._default_timeout, self._owns_kernel, self._client, self._km = timeout, False, None, None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
                   'JUPYTER_PLATFORM_DIRS': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute('import math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

    def _format_error(self, tb):
        return ''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb
                      if 'File "' not in f or 'ipython-input' in f)

    def execute(self, code, timeout=None):
        effective_timeout = timeout or self._default_timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                stderr.append(self._format_error(c.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[WARN] No output.')

    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()

    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')

In [ ]:
class AIMO3Tool:
    def __init__(self, timeout, prompt, sandbox):
        self._jupyter_timeout, self._tool_prompt, self._sandbox = timeout, prompt, sandbox
        self._lock = threading.Lock()

    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if any(x in last for x in ['print', 'import']) or not last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)

    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self._tool_prompt, tools=[])

    def execute(self, code):
        with self._lock:
            return self._sandbox.execute(self._ensure_last_print(code))

## Trace Generator

In [ ]:
class TraceGenerator:
    def __init__(self, cfg, port=8000):
        self.cfg = cfg
        self.port = port
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        
        self._preload_weights()
        self._start_server()
        self.client = OpenAI(base_url=f'http://0.0.0.0:{port}/v1', api_key='sk-local', timeout=600)
        self._wait_for_server()
        self._init_sandbox()
        
    def _preload_weights(self):
        print(f'Preloading model weights from {self.cfg.model_path}...')
        start = time.time()
        files = []
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp): files.append(fp)
        with ThreadPoolExecutor(max_workers=8) as ex:
            list(ex.map(lambda p: open(p, 'rb').read(), files))
        print(f'Preloaded {len(files)} files in {time.time()-start:.1f}s')
        
    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
               '--seed', str(self.cfg.seed), '--model', self.cfg.model_path,
               '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1', '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
               '--host', '0.0.0.0', '--port', str(self.port),
               '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype,
               '--max-model-len', str(self.cfg.context_tokens),
               '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching']
        self.log_file = open('vllm_server.log', 'w')
        self.server = subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)
        
    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if self.server.poll() is not None:
                raise RuntimeError(f'Server died: {open("vllm_server.log").read()}')
            try:
                self.client.models.list()
                print(f'Server ready in {time.time()-start:.1f}s')
                return
            except: time.sleep(1)
        raise RuntimeError('Server timeout')
        
    def _init_sandbox(self):
        print('Initializing sandbox...')
        self.sandbox = AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
        self.tool = AIMO3Tool(self.cfg.jupyter_timeout, self.cfg.tool_prompt, self.sandbox)
        print('Sandbox ready')
        
    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except: pass
        return None
    
    def _compute_entropy(self, logprobs):
        if not logprobs: return float('inf')
        total, count = 0.0, 0
        for lp in logprobs:
            if isinstance(lp, dict) and lp:
                ent = sum(-math.exp(v)*math.log2(math.exp(v)) for v in lp.values() if math.exp(v) > 0)
                total += ent
                count += 1
        return total/count if count else float('inf')
        
    def generate_sample(self, problem_text, sample_idx):
        """Generate a single sample trace."""
        start = time.time()
        self.sandbox.reset()
        
        user_input = f'{problem_text} {self.cfg.preference_prompt}'
        conv = Conversation.from_messages(self.template.apply_chat_template(
            self.cfg.system_prompt, user_input, self.tool.tool_config))
        
        trace = {'turns': [], 'logprobs': []}
        answer = None
        seed = int((self.cfg.seed + sample_idx) ** 2)
        
        for turn in range(self.cfg.max_turns):
            prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
            max_toks = self.cfg.context_tokens - len(prompt_ids)
            if max_toks < self.cfg.buffer_tokens: break
                
            try:
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_toks,
                    prompt=prompt_ids,
                    seed=seed,
                    stream=True,
                    extra_body={'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )
                
                tok_buf, txt_chunks, turn_logprobs = [], [], []
                for chunk in stream:
                    if new_toks := chunk.choices[0].token_ids:
                        tok_buf.extend(new_toks)
                        txt_chunks.append(chunk.choices[0].text)
                        if (clp := chunk.choices[0].logprobs) and clp.top_logprobs:
                            turn_logprobs.extend([dict(lp) for lp in clp.top_logprobs])
                    if '}' in chunk.choices[0].text:
                        if ans := self._scan_for_answer(''.join(txt_chunks[-32:])):
                            answer = ans
                            break
                stream.close()
            except Exception as e:
                trace['error'] = str(e)
                break
                
            if not tok_buf: break
            
            full_text = ''.join(txt_chunks)
            trace['turns'].append({'role': 'assistant', 'content': full_text})
            trace['logprobs'].extend(turn_logprobs)
            
            if answer: break
            
            # Parse and handle tool calls
            new_msgs = self.encoding.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
            conv.messages.extend(new_msgs)
            last = new_msgs[-1]
            
            if last.channel == 'final':
                answer = self._scan_for_answer(last.content[0].text)
                break
                
            if last.recipient == 'python':
                code = last.content[0].text
                output = self.tool.execute(code)
                trace['turns'].append({'role': 'tool', 'name': 'python', 'content': output})
                conv.messages.append(Message(
                    author=Author(role=Role.TOOL, name='python'),
                    content=[TextContent(text=output)]
                ).with_recipient('assistant'))
        
        trace['answer'] = answer
        trace['entropy'] = self._compute_entropy(trace['logprobs'])
        trace['time'] = time.time() - start
        trace['n_turns'] = len([t for t in trace['turns'] if t['role'] == 'assistant'])
        
        return trace
    
    def generate_traces(self, problem):
        """Generate all samples for a problem."""
        problem_id = problem['id']
        problem_text = problem['problem']
        ground_truth = problem.get('answer')
        
        print(f"\n{'='*60}")
        print(f"Problem: {problem_id}")
        print(f"Ground truth: {ground_truth}")
        print(f"{'='*60}")
        
        samples = []
        for i in range(self.cfg.n_samples):
            trace = self.generate_sample(problem_text, i)
            samples.append(trace)
            status = 'OK' if trace['answer'] == ground_truth else ('WRONG' if trace['answer'] else 'NONE')
            print(f"  Sample {i+1}/{self.cfg.n_samples}: answer={trace['answer']} "
                  f"(entropy={trace['entropy']:.3f}, turns={trace['n_turns']}, time={trace['time']:.1f}s) [{status}]")
        
        # Save traces
        output = {
            'problem_id': problem_id,
            'problem': problem_text,
            'ground_truth': ground_truth,
            'samples': samples,
            'model': self.cfg.served_model_name,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        
        output_path = Path(self.cfg.output_dir) / f'problem_{problem_id.replace("-", "_")}.json'
        with open(output_path, 'w') as f:
            json.dump(output, f, indent=2)
        print(f"  Saved to {output_path}")
        
        # Summary
        answers = [s['answer'] for s in samples if s['answer'] is not None]
        if answers:
            from collections import Counter
            votes = Counter(answers)
            top_answer = votes.most_common(1)[0][0]
            correct = top_answer == ground_truth
            print(f"  >> Majority={top_answer}, Votes={dict(votes)}, Correct={correct}")
        
        return output
    
    def cleanup(self):
        self.sandbox.close()
        self.server.terminate()
        self.server.wait()
        self.log_file.close()

## Generate Traces

In [ ]:
generator = TraceGenerator(CFG)

In [ ]:
start_time = time.time()
results = []

for i, problem in enumerate(AIME_PROBLEMS):
    print(f"\n[{i+1}/{len(AIME_PROBLEMS)}] Processing {problem['id']}...")
    result = generator.generate_traces(problem)
    results.append(result)
    
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = len(AIME_PROBLEMS) - i - 1
    eta = avg_time * remaining
    print(f"\nProgress: {i+1}/{len(AIME_PROBLEMS)} | Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min")

print(f"\n{'='*60}")
print(f"DONE! Generated traces for {len(results)} problems")
print(f"Total time: {(time.time()-start_time)/60:.1f} minutes")
print(f"Output dir: {CFG.output_dir}")

In [ ]:
# List output files
!ls -la /kaggle/working/traces/

In [ ]:
generator.cleanup()
print("Cleanup complete")